In [12]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import random
import seaborn as sns
import scipy
import sklearn
from sklearn.svm import SVC
from scipy.stats import pearsonr
from sklearn import datasets, linear_model
from sklearn import preprocessing
from sklearn.pipeline import make_pipeline
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.model_selection import (
    train_test_split, 
    StratifiedKFold, cross_val_score,  
    RepeatedStratifiedKFold, 
    RandomizedSearchCV,
    train_test_split, 
    KFold
)
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import mean_squared_error, accuracy_score, classification_report, confusion_matrix
from sklearn.svm import SVR
from sklearn.tree import DecisionTreeRegressor, plot_tree, DecisionTreeClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve,
    f1_score
)
from sklearn.feature_selection import SelectFromModel
from sklearn.compose import ColumnTransformer
from scipy.stats import loguniform

#!pip install xlrd 
#!pip install category_encoders
import category_encoders as ce



   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.5/10.5 MB 13.8 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [category_encoders]statsmodels]


In [14]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()

,ID,pCR (outcome),RelapseFreeSurvival (outcome),Age,ER,PgR,HER2,TrippleNegative,ChemoGrade,Proliferation,...,original_glszm_SmallAreaHighGrayLevelEmphasis,original_glszm_SmallAreaLowGrayLevelEmphasis,original_glszm_ZoneEntropy,original_glszm_ZonePercentage,original_glszm_ZoneVariance,original_ngtdm_Busyness,original_ngtdm_Coarseness,original_ngtdm_Complexity,original_ngtdm_Contrast,original_ngtdm_Strength
0,TRG002174,1,144.0,41.0,0,0,0,1,3,3,...,0.517172,0.375126,3.325332,0.002314,3880771.500,473.464852,0.000768,0.182615,0.030508,0.000758
1,TRG002178,0,142.0,39.0,1,1,0,0,3,3,...,0.444391,0.444391,3.032144,0.005612,2372009.744,59.459710,0.004383,0.032012,0.001006,0.003685
2,TRG002204,1,135.0,31.0,0,0,0,1,2,1,...,0.534549,0.534549,2.485848,0.006752,1540027.421,33.935384,0.007584,0.024062,0.000529,0.006447
3,TRG002206,0,12.0,35.0,0,0,0,1,3,3,...,0.506185,0.506185,2.606255,0.003755,6936740.794,46.859265,0.005424,0.013707,0.000178,0.004543
4,TRG002210,0,109.0,61.0,1,0,0,0,2,1,...,0.462282,0.462282,2.809279,0.006521,1265399.054,39.621023,0.006585,0.034148,0.001083,0.005626


In [ ]:
df.info()
df.describe()

In [ ]:
df.replace(999, pd.NA, inplace=True)

In [ ]:
missing_cols = df.isna().sum()
missing_cols = missing_cols[missing_cols > 0]
missing_cols

In [ ]:
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

print(summary)

In [ ]:
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape

In [ ]:
missing_cols = df_dropped.columns[df_dropped.isna().any()]
# Missing data summary
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df_dropped[col].isna().sum() for col in missing_cols],
    'Dtype': [df_dropped[col].dtype for col in missing_cols]
})

print(summary)

In [ ]:
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
# Separate target
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum())

In [ ]:
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import SelectFromModel
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
    precision_recall_curve,
    roc_curve
)

# -----------------------------
# Build X, y
# -----------------------------
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

#print("Number of numeric features:", len(num_features))
#print("Categorical features:", cat_features)

# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)


In [ ]:
# Preprocessing and Feature Selection
'''
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Preprocess: scale numeric, one-hot encode categoricals
numeric_transformer = StandardScaler()
categorical_transformer = OneHotEncoder(handle_unknown="ignore")

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, num_features),
        ("cat", categorical_transformer, cat_features),
    ]
)
'''


In [ ]:
import numpy as np
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.ensemble import RandomForestClassifier

class RFPercentileSelector(BaseEstimator, TransformerMixin):
    """
    Select features whose RandomForest importance is above a given percentile.
    Works as a drop-in transformer inside a Pipeline.
    """
    def __init__(
        self,
        percentile=50,
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ):
        self.percentile = percentile
        self.n_estimators = n_estimators
        self.max_depth = max_depth
        self.min_samples_leaf = min_samples_leaf
        self.n_jobs = n_jobs
        self.random_state = random_state
        self.class_weight = class_weight

    def fit(self, X, y):
        # Train RF on the *preprocessed* matrix X
        self.rf_ = RandomForestClassifier(
            n_estimators=self.n_estimators,
            max_depth=self.max_depth,
            min_samples_leaf=self.min_samples_leaf,
            n_jobs=self.n_jobs,
            random_state=self.random_state,
            class_weight=self.class_weight,
        )
        self.rf_.fit(X, y)

        importances = self.rf_.feature_importances_
        self.threshold_ = np.percentile(importances, self.percentile)
        self.mask_ = importances >= self.threshold_

        # For compatibility with some sklearn utilities
        self.feature_importances_ = importances
        return self

    def transform(self, X):
        return X[:, self.mask_]


In [ ]:
keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)

rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean
)


In [ ]:
from sklearn.svm import SVC
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV
from scipy.stats import loguniform

# Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        #("select", rf_selector),
        ("select", RFPercentileSelector(percentile=45)),
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=100,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_


In [ ]:
'''
# Random Forest used as feature selector
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=None,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced"  # helps with pCR imbalance
    ),
    threshold="median"   # keep features with importance above median, can also try mean
)
'''

In [ ]:
'''
# SVM Pipeline and Hyperparameter Search
from scipy.stats import loguniform
from sklearn.model_selection import RepeatedStratifiedKFold, RandomizedSearchCV

# Full pipeline: preprocessing -> RF feature selection -> SVM
svm_pipe = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("select", rf_selector),
    ("svm", SVC(kernel="rbf", probability=False))
])

# Repeated stratified CV – good for small, imbalanced datasets
cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42
)

# Hyperparameter search space
param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,              # can increase if compute allows
    cv=cv,
    scoring="roc_auc",      # key metric for this task
    n_jobs=-1,
    random_state=42,
    verbose=1
)

search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_
comments = 
Do the best C and gamma sit near the edges of your search range?
If yes, extend the range (e.g. 1e-4 to 1e4).
'''

In [ ]:
# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)

# Basic metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))


In [ ]:
preprocessor = best_svm.named_steps["preprocess"]
selector = best_svm.named_steps["select"]

# Names of all features *after* preprocessing (including one-hot columns)
all_feature_names = preprocessor.get_feature_names_out()
#print(all_feature_names)
print("Total transformed features:", len(all_feature_names))

In [ ]:
mask = selector.get_support()  # True = selected, False = dropped
selected_features = np.array(all_feature_names)[mask]
dropped_features  = np.array(all_feature_names)[~mask]

print("Selected features:", len(selected_features))
print("Dropped features:", len(dropped_features))

print("Some selected features:")
for f in selected_features[:20]:
    print("  ", f)

print("\nSome dropped features:")
for f in dropped_features[:20]:
    print("  ", f)


In [ ]:
# Threshold tuning
import matplotlib.pyplot as plt

# Compute ROC and PR curves
fpr, tpr, roc_thresholds = roc_curve(y_test, scores_test)
prec, rec, pr_thresholds = precision_recall_curve(y_test, scores_test)

# Example: try a grid of thresholds and compute metrics
thresholds = np.linspace(scores_test.min(), scores_test.max(), 200)

results = []
for thr in thresholds:
    y_pred_thr = (scores_test >= thr).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test, y_pred_thr).ravel()
    
    # Avoid division by zero
    sens = tp / (tp + fn) if (tp + fn) > 0 else 0.0  # recall for pCR=1
    spec = tn / (tn + fp) if (tn + fp) > 0 else 0.0  # specificity
    bal_acc_thr = 0.5 * (sens + spec)
    
    results.append((thr, sens, spec, bal_acc_thr))

thr_arr = np.array([r[0] for r in results])
sens_arr = np.array([r[1] for r in results])
spec_arr = np.array([r[2] for r in results])
bal_arr  = np.array([r[3] for r in results])

# Example: pick the threshold that maximises balanced accuracy
best_idx = np.argmax(bal_arr)
best_thr = thr_arr[best_idx]
print("Best threshold by balanced accuracy:", best_thr)
print("Balanced accuracy at this threshold:", bal_arr[best_idx])
print("Sensitivity (recall for pCR=1):", sens_arr[best_idx])
print("Specificity:", spec_arr[best_idx])

# Predictions with best threshold
y_pred_best = (scores_test >= best_thr).astype(int)
print("Confusion matrix (best threshold):\n", confusion_matrix(y_test, y_pred_best))
print("Classification report (best threshold):\n", classification_report(y_test, y_pred_best))

# Optional: visualise sensitivity and specificity vs threshold
plt.figure(figsize=(8, 5))
plt.plot(thr_arr, sens_arr, label="Sensitivity (recall class 1)")
plt.plot(thr_arr, spec_arr, label="Specificity (class 0)")
plt.plot(thr_arr, bal_arr, label="Balanced accuracy")
plt.xlabel("Decision threshold")
plt.ylabel("Metric value")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.show()


In [ ]:
final_model = {
    "pipeline": best_svm,
    "threshold": best_thr
}


In [ ]:
def svm_predict(X_test):
    scores_test = final_model["pipeline"].decision_function(X_test)  # continuous margins
    y_pred_default = final_model["pipeline"].predict(X_test)
    y_pred_best = (scores_test >= best_thr).astype(int)
    return y_pred_best

In [ ]:
svm_predict(X_test)

In [15]:
df = pd.read_excel("TrainDataset2025.xls")
df.head()
df.replace(999, pd.NA, inplace=True)
# Get columns with missing values
missing_cols = df.columns[df.isna().any()]

# Missing data summary before droppig "pCR" missing values rows and "RelapseFreeSurvival" column
summary = pd.DataFrame({
    'Column': missing_cols,
    'MissingCount': [df[col].isna().sum() for col in missing_cols],
    'Dtype': [df[col].dtype for col in missing_cols]
})

summary
df_dropped =  df.dropna(subset=['pCR (outcome)'])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)
df_dropped.shape
# Iterative imputation

df_imputed = df_dropped.copy()

# Select categorical columns
cat_columns = df_imputed.select_dtypes(include=['object', 'category']).columns
cat_cols_to_encode = cat_columns.drop('pCR (outcome)')

encoder = ce.OrdinalEncoder(handle_missing='return_nan') 


# Encode categorical features
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])


## Imputation with IterativeImputer
y_target = df_imputed['pCR (outcome)']
X_features = df_imputed.drop(columns=['pCR (outcome)'])

# Iterative Imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Convert back to DataFrame
X_imputed_df = pd.DataFrame(X_imputed,
                            columns=X_features.columns,
                            index=X_features.index)

# Round encoded categorical columns back to integers
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

    
# Inverse transform encoded categorical columns
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])


# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]
print("\n--- Missing Value Check ---")
print(final_df.isna().sum().sum())



y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()


# Train / test split (hold out 30% for final evaluation)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

keep_cats = ["Gene", "HER2", "ER"] 
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop"
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop"
)




--- Missing Value Check ---
0


In [59]:
# Features = [ER/HER2/Gene one-hot] + [RF-selected other features]
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector), 
    ])),
])

svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)


param_dist = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)


search.fit(X_train, y_train)

print("Best params:", search.best_params_)
print("Best CV ROC AUC:", search.best_score_)

best_svm = search.best_estimator_

# Evaluation of Model
# Fit best pipeline on full training set
best_svm.fit(X_train, y_train)

# Decision scores on test set
scores_test = best_svm.decision_function(X_test)  # continuous margins
y_pred_default = best_svm.predict(X_test)
# y_pred_best = (scores_test >= thr_bal).astype(int)

# Metrics with default decision threshold
roc_auc = roc_auc_score(y_test, scores_test)
pr_auc = average_precision_score(y_test, scores_test)  # PR AUC
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix (default threshold):\n", confusion_matrix(y_test, y_pred_default))
print("Classification report (default threshold):\n", classification_report(y_test, y_pred_default))

Fitting 25 folds for each of 50 candidates, totalling 1250 fits
Best params: {'svm__C': np.float64(361.2478500429091), 'svm__class_weight': 'balanced', 'svm__gamma': np.float64(0.00037961668958008145)}
Best CV ROC AUC: 0.6903497981933501
Test ROC AUC: 0.7638297872340426
Test PR AUC: 0.5205332717241044
Test balanced accuracy: 0.7257446808510639
Confusion matrix (default threshold):
 [[65 29]
 [ 6 19]]
Classification report (default threshold):
               precision    recall  f1-score   support

           0       0.92      0.69      0.79        94
           1       0.40      0.76      0.52        25

    accuracy                           0.71       119
   macro avg       0.66      0.73      0.65       119
weighted avg       0.81      0.71      0.73       119



In [60]:
rf_selector = SelectFromModel(
    RandomForestClassifier(
        n_estimators=500,
        max_depth=20,
        min_samples_leaf=3,
        n_jobs=-1,
        random_state=42,
        class_weight="balanced",
    ),
    threshold="median", # mean better result using median
)

from xgboost import XGBClassifier
from sklearn.feature_selection import SelectFromModel

xgb_selector = SelectFromModel( # poor result
    XGBClassifier(
        n_estimators=300,
        max_depth=4,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        scale_pos_weight=1.5,  # imbalance handling
        random_state=42,
        n_jobs=-1,
        eval_metric="logloss"
    ),
    threshold="median"
)

from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import SelectFromModel

logreg_l1_selector = SelectFromModel( # poor result
    LogisticRegression(
        penalty="l1",
        solver="saga",
        class_weight="balanced",
        random_state=42,
        max_iter=5000,
    ),
    threshold="median"  # or "mean", or "0.5*median"
)


from boruta import BorutaPy
from sklearn.ensemble import RandomForestClassifier

boruta_selector = BorutaPy( # slow and poor result
    estimator=RandomForestClassifier(
        n_estimators=1000, 
        class_weight="balanced", 
        random_state=42,
        n_jobs=-1),
    n_estimators="auto",
    max_iter=50
)

from sklearn.feature_selection import RFE
from sklearn.svm import LinearSVC

rfe_selector = RFE(
    estimator=LinearSVC(
        penalty="l2",
        class_weight="balanced",
        random_state=42,
    ),
    n_features_to_select=42,   # tune based on performance
    step=0.1
)

In [63]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import train_test_split, RepeatedStratifiedKFold, RandomizedSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        #("select", rf_selector),
        ("select", boruta_selector),
    ])),
])

# -----------------------------
# 5. Random Forest classifier + tuning
# -----------------------------
rf_clf = RandomForestClassifier(random_state=42)

rf_pipe = Pipeline(steps=[
    ("features", full_features),
    ("rf", rf_clf),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

# Hyperparameter search space (advanced RF)
param_dist_rf = {
    "rf__n_estimators": [100, 200, 300, 500, 600],
    "rf__max_depth": [None, 3, 5, 7, 9, 11, 15],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2", 0.3, 0.5, None],
    "rf__bootstrap": [True, False],
    "rf__class_weight": [None, "balanced", "balanced_subsample"],
}

search_rf = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist_rf,
    n_iter=60,               # adjust if too slow
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for RandomForest...")
search_rf.fit(X_train, y_train)

print("\nBest RF params:", search_rf.best_params_)
print("Best CV ROC AUC:", search_rf.best_score_)

best_rf = search_rf.best_estimator_

# -----------------------------
# 6. Final evaluation on test set
# -----------------------------
best_rf.fit(X_train, y_train)

# For RF we use predict_proba for continuous scores
proba_test = best_rf.predict_proba(X_test)[:, 1]
y_pred_default = best_rf.predict(X_test)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- Random Forest Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))



Fitting RandomizedSearchCV for RandomForest...
Fitting 25 folds for each of 60 candidates, totalling 1500 fits

Best RF params: {'rf__n_estimators': 100, 'rf__min_samples_split': 2, 'rf__min_samples_leaf': 4, 'rf__max_features': 0.3, 'rf__max_depth': None, 'rf__class_weight': 'balanced', 'rf__bootstrap': False}
Best CV ROC AUC: 0.7493411653533218

--- Random Forest Test Performance (default threshold 0.5) ---
Test ROC AUC: 0.7117021276595745
Test PR AUC: 0.4022501939183908
Test balanced accuracy: 0.5914893617021277
Confusion matrix:
 [[36 58]
 [ 5 20]]
Classification report:
               precision    recall  f1-score   support

           0       0.88      0.38      0.53        94
           1       0.26      0.80      0.39        25

    accuracy                           0.47       119
   macro avg       0.57      0.59      0.46       119
weighted avg       0.75      0.47      0.50       119



In [24]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

from sklearn.utils.class_weight import compute_class_weight

from scipy import sparse


# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        #("select", rf_selector),
        ("select", rf_selector),
    ])),
])

# -----------------------------
# 5. Fit preprocessing + feature selection
# -----------------------------
print("\nFitting preprocessing + RF feature selection...")
full_features.fit(X_train, y_train)

X_train_proc = full_features.transform(X_train)
X_test_proc = full_features.transform(X_test)

# Convert sparse to dense for Keras
if sparse.issparse(X_train_proc):
    X_train_proc = X_train_proc.toarray()
if sparse.issparse(X_test_proc):
    X_test_proc = X_test_proc.toarray()

print("Processed train shape:", X_train_proc.shape)
print("Processed test shape:", X_test_proc.shape)

# -----------------------------
# 6. Build ANN model in TensorFlow / Keras
# -----------------------------
input_dim = X_train_proc.shape[1]

def build_ann_model(input_dim: int) -> keras.Model:
    model = keras.Sequential([
        layers.Input(shape=(input_dim,)),
        layers.Dense(128, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(64, activation="relu"),
        layers.BatchNormalization(),
        layers.Dropout(0.3),

        layers.Dense(32, activation="relu"),
        layers.BatchNormalization(),

        layers.Dense(1, activation="sigmoid"),  # binary classification
    ])

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=1e-3),
        loss="binary_crossentropy",
        metrics=[
            keras.metrics.AUC(name="auc"),
            keras.metrics.AUC(name="pr_auc", curve="PR"),
            keras.metrics.BinaryAccuracy(name="accuracy"),
        ],
    )
    return model

model = build_ann_model(input_dim)

# -----------------------------
# 7. Class weights + callbacks
# -----------------------------
classes = np.unique(y_train)
class_weights_array = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train,
)
class_weight_dict = {cls: w for cls, w in zip(classes, class_weights_array)}
print("\nClass weights:", class_weight_dict)

early_stopping = keras.callbacks.EarlyStopping(
    monitor="val_auc",
    mode="max",
    patience=15,
    restore_best_weights=True,
    verbose=1,
)

reduce_lr = keras.callbacks.ReduceLROnPlateau(
    monitor="val_auc",
    mode="max",
    factor=0.5,
    patience=5,
    min_lr=1e-6,
    verbose=1,
)

# -----------------------------
# 8. Train ANN (with validation split)
# -----------------------------
history = model.fit(
    X_train_proc,
    y_train,
    epochs=200,
    batch_size=32,
    validation_split=0.2,
    callbacks=[early_stopping, reduce_lr],
    class_weight=class_weight_dict,
    verbose=1,
)

# -----------------------------
# 9. Evaluation on test set
# -----------------------------
# Probabilities (for class 1)
proba_test = model.predict(X_test_proc).ravel()
y_pred_default = (proba_test >= 0.5).astype(int)  # default threshold 0.5

roc_auc = roc_auc_score(y_test, proba_test)
pr_auc = average_precision_score(y_test, proba_test)
bal_acc = balanced_accuracy_score(y_test, y_pred_default)

print("\n--- ANN Test Performance (default threshold 0.5) ---")
print("Test ROC AUC:", roc_auc)
print("Test PR AUC:", pr_auc)
print("Test balanced accuracy:", bal_acc)
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default))
print("Classification report:\n", classification_report(y_test, y_pred_default))


Shape after dropping pCR-missing and RFS: (395, 120)

--- Missing Value Check ---
Total missing values: 0

Numeric features: 110
Categorical features: ['ChemoGrade', 'Gene', 'HER2', 'HistologyType', 'LNStatus', 'PgR', 'Proliferation', 'TrippleNegative']

Fitting preprocessing + RF feature selection...
Processed train shape: (276, 68)
Processed test shape: (119, 68)

Class weights: {np.int64(0): np.float64(0.6359447004608295), np.int64(1): np.float64(2.3389830508474576)}
Epoch 1/200


2025-12-04 01:07:39.279884: E external/local_xla/xla/stream_executor/cuda/cuda_platform.cc:51] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


7/7 - 2s - 262ms/step - accuracy: 0.4727 - auc: 0.4955 - loss: 0.9413 - pr_auc: 0.2031 - val_accuracy: 0.6250 - val_auc: 0.6391 - val_loss: 0.6442 - val_pr_auc: 0.3522 - learning_rate: 0.0010
Epoch 2/200
7/7 - 0s - 11ms/step - accuracy: 0.5364 - auc: 0.6364 - loss: 0.7283 - pr_auc: 0.3711 - val_accuracy: 0.6607 - val_auc: 0.6625 - val_loss: 0.6567 - val_pr_auc: 0.3604 - learning_rate: 0.0010
Epoch 3/200
7/7 - 0s - 10ms/step - accuracy: 0.5545 - auc: 0.6508 - loss: 0.6819 - pr_auc: 0.4161 - val_accuracy: 0.6607 - val_auc: 0.6617 - val_loss: 0.6659 - val_pr_auc: 0.3644 - learning_rate: 0.0010
Epoch 4/200
7/7 - 0s - 10ms/step - accuracy: 0.6409 - auc: 0.7174 - loss: 0.6349 - pr_auc: 0.3228 - val_accuracy: 0.6786 - val_auc: 0.6641 - val_loss: 0.6655 - val_pr_auc: 0.3676 - learning_rate: 0.0010
Epoch 5/200
7/7 - 0s - 11ms/step - accuracy: 0.6727 - auc: 0.7645 - loss: 0.5735 - pr_auc: 0.4243 - val_accuracy: 0.6607 - val_auc: 0.6711 - val_loss: 0.6611 - val_pr_auc: 0.3743 - learning_rate: 0.0

In [ ]:
import pandas as pd
import numpy as np

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)

from scipy.stats import loguniform

np.random.seed(42)

# =========================================================
# 1. Load data and initial cleaning
# =========================================================
df = pd.read_excel("TrainDataset2025.xls")

# Replace sentinel missing code with NA
df.replace(999, pd.NA, inplace=True)

# Drop rows with missing pCR and drop RFS outcome column
df_dropped = df.dropna(subset=["pCR (outcome)"])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)

print("Shape after dropping pCR-missing and RFS:", df_dropped.shape)

# =========================================================
# 2. Iterative imputation with ordinal-encoded categoricals
# =========================================================
df_imputed = df_dropped.copy()

# Categorical columns
cat_columns = df_imputed.select_dtypes(include=["object", "category"]).columns
cat_cols_to_encode = cat_columns.drop("pCR (outcome)")

# Ordinal encoder for imputation
encoder = ce.OrdinalEncoder(handle_missing="return_nan")

df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])

# Separate target and features
y_target = df_imputed["pCR (outcome)"]
X_features = df_imputed.drop(columns=["pCR (outcome)"])

# Iterative imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Back to DataFrame
X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X_features.columns,
    index=X_features.index,
)

# Round encoded categorical columns and clip to original range
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

# Inverse transform categorical columns back to original labels
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])

# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]  # keep columns sorted

print("\n--- Missing Value Check ---")
print("Total missing values:", final_df.isna().sum().sum())

# =========================================================
# 3. Train/test split and feature groups
# =========================================================
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)  # drop ID from features

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", len(num_features))
print("Categorical features:", cat_features)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# =========================================================
# 4. RF-based feature selection block (same as RF pipeline)
# =========================================================
rf_selector_estimator = RandomForestClassifier(
    n_estimators=500,
    max_depth=None,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",   # or "mean"
    prefit=False,
)

# Combine "always keep" block with RF-selected block
full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# =========================================================
# 5. MLPClassifier (ANN-style) + RandomizedSearchCV
# =========================================================
mlp = MLPClassifier(
    max_iter=1000,
    random_state=42,
)

mlp_pipe = Pipeline(steps=[
    ("features", full_features),
    ("mlp", mlp),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_mlp = {
    "mlp__hidden_layer_sizes": [
        (128, 64, 32),
        (64, 32),
        (128,),
        (64,),
    ],
    "mlp__alpha": loguniform(1e-5, 1e-2),          # L2 regularisation
    "mlp__learning_rate_init": loguniform(1e-4, 1e-2),
    "mlp__activation": ["relu", "tanh"],
    # solver is kept as default "adam" (works well, supports early-stopping if you want)
}

search_mlp = RandomizedSearchCV(
    estimator=mlp_pipe,
    param_distributions=param_dist_mlp,
    n_iter=40,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for MLP (ANN-style)...")
search_mlp.fit(X_train, y_train)

print("\nBest MLP params:", search_mlp.best_params_)
print("Best CV ROC AUC (MLP):", search_mlp.best_score_)

best_mlp = search_mlp.best_estimator_

# =========================================================
# 6. Threshold tuning helper
# =========================================================
def tune_threshold(proba: np.ndarray, y_true: np.ndarray):
    """
    Scan thresholds in [0.01, 0.99] and pick the one that maximises
    balanced accuracy (equivalently Youden's J).
    """
    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr = 0.5
    best_bal = -1.0

    for thr in thresholds:
        y_pred = (proba >= thr).astype(int)
        bal = balanced_accuracy_score(y_true, y_pred)
        if bal > best_bal:
            best_bal = bal
            best_thr = thr

    return best_thr, best_bal

# =========================================================
# 7. Evaluation on test set (default and tuned threshold)
# =========================================================
# Fit best model on full training data
best_mlp.fit(X_train, y_train)

# Probabilities for class 1
proba_test_mlp = best_mlp.predict_proba(X_test)[:, 1]

# Default threshold 0.5
y_pred_default_mlp = (proba_test_mlp >= 0.5).astype(int)

# Tuned threshold based on test set (if you prefer, you can split a validation set instead)
best_thr, best_bal = tune_threshold(proba_test_mlp, y_test)

y_pred_best_mlp = (proba_test_mlp >= best_thr).astype(int)

print("\n=== MLP Test Performance (default threshold 0.5) ===")
print("Test ROC AUC:", roc_auc_score(y_test, proba_test_mlp))
print("Test PR AUC:", average_precision_score(y_test, proba_test_mlp))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_pred_default_mlp))
print("Confusion matrix:\n", confusion_matrix(y_test, y_pred_default_mlp))
print("Classification report:\n", classification_report(y_test, y_pred_default_mlp))

print("\n=== MLP Test Performance (tuned threshold) ===")
print("Tuned threshold:", best_thr)
print("Test balanced accuracy (tuned):", balanced_accuracy_score(y_test, y_pred_best_mlp))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_pred_best_mlp))
print("Classification report (tuned):\n", classification_report(y_test, y_pred_best_mlp))


In [64]:
import numpy as np
import pandas as pd

# -----------------------------
# Imputation + Encoding imports
# -----------------------------
from sklearn.experimental import enable_iterative_imputer  # noqa: F401
from sklearn.impute import IterativeImputer
import category_encoders as ce

# -----------------------------
# Sklearn pipeline imports
# -----------------------------
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier

from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)
from scipy.stats import loguniform

np.random.seed(42)

# =========================================================
# 1. Load data and initial cleaning
# =========================================================
df = pd.read_excel("TrainDataset2025.xls")

# Replace sentinel missing code with NA
df.replace(999, pd.NA, inplace=True)

# Drop rows with missing pCR and drop RFS outcome column
df_dropped = df.dropna(subset=["pCR (outcome)"])
df_dropped = df_dropped.drop("RelapseFreeSurvival (outcome)", axis=1)

print("Shape after dropping pCR-missing and RFS:", df_dropped.shape)

# =========================================================
# 2. Iterative imputation with ordinal-encoded categoricals
# =========================================================
df_imputed = df_dropped.copy()

# Categorical columns
cat_columns = df_imputed.select_dtypes(include=["object", "category"]).columns
cat_cols_to_encode = cat_columns.drop("pCR (outcome)")

# Ordinal encoder for imputation
encoder = ce.OrdinalEncoder(handle_missing="return_nan")
df_imputed[cat_cols_to_encode] = encoder.fit_transform(df_imputed[cat_cols_to_encode])

# Separate target and features
y_target = df_imputed["pCR (outcome)"]
X_features = df_imputed.drop(columns=["pCR (outcome)"])

# Iterative imputer
imputer = IterativeImputer(max_iter=20, random_state=42)
X_imputed = imputer.fit_transform(X_features)

# Back to DataFrame
X_imputed_df = pd.DataFrame(
    X_imputed,
    columns=X_features.columns,
    index=X_features.index,
)

# Round encoded categorical columns and clip to original range
for col in cat_cols_to_encode:
    max_val = df_imputed[col].max()
    min_val = df_imputed[col].min()
    X_imputed_df[col] = X_imputed_df[col].clip(lower=min_val, upper=max_val)
    X_imputed_df[col] = X_imputed_df[col].round().astype(int)

# Inverse transform categorical columns back to original labels
X_imputed_df[cat_cols_to_encode] = encoder.inverse_transform(X_imputed_df[cat_cols_to_encode])

# Recombine target
final_df = pd.concat([X_imputed_df, y_target], axis=1)
final_df = final_df[sorted(final_df.columns)]

print("\n--- Missing Value Check ---")
print("Total missing values:", final_df.isna().sum().sum())

# =========================================================
# 3. Train/test split and feature groups
# =========================================================
y = final_df["pCR (outcome)"].astype(int)
X = final_df.drop(["pCR (outcome)", "ID"], axis=1)

num_features = X.select_dtypes(include=["int64", "float64"]).columns.tolist()
cat_features = X.select_dtypes(include=["object", "category"]).columns.tolist()

print("\nNumeric features:", len(num_features))
print("Categorical features:", cat_features)

# Outer train/test split (test is held out to the very end)
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    stratify=y,
    random_state=42,
)

# Inner train/validation split (for threshold tuning)
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# =========================================================
# 4. RF-based feature selection block
# =========================================================
rf_selector_estimator = RandomForestClassifier(
    n_estimators=300,        # slightly lighter than 500; adjust if you like
    max_depth=8,            # depth limit for more stable importance
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",     # or "mean" for fewer, stronger features
    prefit=False,
)

full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# =========================================================
# 5. Random Forest classifier + hyperparameter tuning
#    (on inner training data only)
# =========================================================
rf_clf = RandomForestClassifier(random_state=42)

rf_pipe = Pipeline(steps=[
    ("features", full_features),
    ("rf", rf_clf),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_rf = {
    "rf__n_estimators": [200, 400, 600, 800, 1000],
    "rf__max_depth": [5, 8, 11, None],
    "rf__min_samples_split": [2, 5, 10],
    "rf__min_samples_leaf": [1, 2, 4],
    "rf__max_features": ["sqrt", "log2", 0.5, None],
    "rf__bootstrap": [True, False],
    "rf__class_weight": [None, "balanced", "balanced_subsample"],
}

search_rf = RandomizedSearchCV(
    estimator=rf_pipe,
    param_distributions=param_dist_rf,
    n_iter=40,          # adjust for time
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for RF on inner training data...")
search_rf.fit(X_train_inner, y_train_inner)

print("\nBest RF params:", search_rf.best_params_)
print("Best inner CV ROC AUC:", search_rf.best_score_)

best_rf_inner = search_rf.best_estimator_

# =========================================================
# 6. Threshold tuning on validation data (no test leakage)
# =========================================================
def tune_threshold(proba, y_true):
    """
    Scan thresholds in [0.01, 0.99] and pick the one
    that maximises balanced accuracy (Youden's J).
    """
    proba = np.asarray(proba)
    y_true = np.asarray(y_true)

    thresholds = np.linspace(0.01, 0.99, 99)
    best_thr = 0.5
    best_bal = -1.0

    for thr in thresholds:
        y_pred = (proba >= thr).astype(int)
        bal = balanced_accuracy_score(y_true, y_pred)
        if bal > best_bal:
            best_bal = bal
            best_thr = thr

    return best_thr, best_bal

# Probabilities on validation set
proba_val = best_rf_inner.predict_proba(X_val)[:, 1]
best_thr, best_bal_val = tune_threshold(proba_val, y_val)

print(f"\nBest threshold on validation (by balanced accuracy): {best_thr:.3f}")
print(f"Validation balanced accuracy at best threshold: {best_bal_val:.4f}")

# Optional: show full validation metrics at tuned threshold
y_val_pred_best = (proba_val >= best_thr).astype(int)
print("\n--- Validation metrics at tuned threshold ---")
print("ROC AUC (val):", roc_auc_score(y_val, proba_val))
print("PR AUC (val):", average_precision_score(y_val, proba_val))
print("Balanced accuracy (val):", balanced_accuracy_score(y_val, y_val_pred_best))
print("Confusion matrix (val):\n", confusion_matrix(y_val, y_val_pred_best))
print("Classification report (val):\n", classification_report(y_val, y_val_pred_best))

# =========================================================
# 7. Refit best RF on full training data (inner + val)
#    and evaluate on the untouched test set
# =========================================================
best_rf_final = search_rf.best_estimator_
best_rf_final.fit(X_train, y_train)   # X_train = inner + val

# Probabilities on test
proba_test = best_rf_final.predict_proba(X_test)[:, 1]

# Default threshold 0.5
y_test_pred_default = (proba_test >= 0.5).astype(int)

# Tuned threshold from validation
y_test_pred_tuned = (proba_test >= best_thr).astype(int)

print("\n=== RF Test Performance (default threshold 0.5) ===")
print("Test ROC AUC:", roc_auc_score(y_test, proba_test))
print("Test PR AUC:", average_precision_score(y_test, proba_test))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_default))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred_default))
print("Classification report:\n", classification_report(y_test, y_test_pred_default))

print("\n=== RF Test Performance (tuned threshold from validation) ===")
print("Threshold used:", best_thr)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_tuned))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_test_pred_tuned))
print("Classification report (tuned):\n", classification_report(y_test, y_test_pred_tuned))


Shape after dropping pCR-missing and RFS: (395, 120)

--- Missing Value Check ---
Total missing values: 0

Numeric features: 110
Categorical features: ['ChemoGrade', 'Gene', 'HER2', 'HistologyType', 'LNStatus', 'PgR', 'Proliferation', 'TrippleNegative']

Fitting RandomizedSearchCV for RF on inner training data...
Fitting 25 folds for each of 40 candidates, totalling 1000 fits

Best RF params: {'rf__n_estimators': 800, 'rf__min_samples_split': 10, 'rf__min_samples_leaf': 4, 'rf__max_features': 'log2', 'rf__max_depth': 5, 'rf__class_weight': 'balanced_subsample', 'rf__bootstrap': True}
Best inner CV ROC AUC: 0.66668720821662

Best threshold on validation (by balanced accuracy): 0.300
Validation balanced accuracy at best threshold: 0.7121

--- Validation metrics at tuned threshold ---
ROC AUC (val): 0.7291666666666666
PR AUC (val): 0.37604001096562656
Balanced accuracy (val): 0.7121212121212122
Confusion matrix (val):
 [[26 18]
 [ 2 10]]
Classification report (val):
               precisi

In [65]:
import numpy as np
from sklearn.svm import SVC
from sklearn.model_selection import (
    train_test_split,
    RepeatedStratifiedKFold,
    RandomizedSearchCV,
)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.metrics import (
    roc_auc_score,
    average_precision_score,
    balanced_accuracy_score,
    confusion_matrix,
    classification_report,
)
from sklearn.feature_selection import SelectFromModel
from sklearn.ensemble import RandomForestClassifier
from scipy.stats import loguniform

# =========================================================
# 1. (If not already defined) preprocessing + feature selector
#    Reuse these if they already exist in your RF script
# =========================================================

# Categorical to always keep (ER/HER2/Gene) vs others
keep_cats = ["Gene", "HER2", "ER"]
other_cats = [c for c in cat_features if c not in keep_cats]

keep_preprocess = ColumnTransformer(
    transformers=[
        ("keep_cat", OneHotEncoder(handle_unknown="ignore"), keep_cats),
    ],
    remainder="drop",
)

other_preprocess = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_features),
        ("cat", OneHotEncoder(handle_unknown="ignore"), other_cats),
    ],
    remainder="drop",
)

# RF selector just for feature selection (same logic as RF pipeline)
rf_selector_estimator = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=3,
    n_jobs=-1,
    random_state=42,
    class_weight="balanced",
)

rf_selector = SelectFromModel(
    estimator=rf_selector_estimator,
    threshold="median",   # or "mean"
    prefit=False,
)

full_features = FeatureUnion([
    ("always_keep", keep_preprocess),
    ("rf_selected", Pipeline([
        ("prep", other_preprocess),
        ("select", rf_selector),
    ])),
])

# =========================================================
# 2. Inner train/validation split for threshold tuning
# =========================================================
X_train_inner, X_val, y_train_inner, y_val = train_test_split(
    X_train,
    y_train,
    test_size=0.2,
    stratify=y_train,
    random_state=42,
)

# =========================================================
# 3. SVM pipeline + hyperparameter tuning on inner train
# =========================================================
svm_pipe = Pipeline(steps=[
    ("features", full_features),
    ("svm", SVC(kernel="rbf", probability=False)),
])

cv = RepeatedStratifiedKFold(
    n_splits=5,
    n_repeats=5,
    random_state=42,
)

param_dist_svm = {
    "svm__C": loguniform(1e-3, 1e3),
    "svm__gamma": loguniform(1e-4, 1e1),
    "svm__class_weight": [None, "balanced"],
}

search_svm = RandomizedSearchCV(
    estimator=svm_pipe,
    param_distributions=param_dist_svm,
    n_iter=50,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
    random_state=42,
    verbose=1,
)

print("\nFitting RandomizedSearchCV for SVM on inner training data...")
search_svm.fit(X_train_inner, y_train_inner)

print("\nBest SVM params:", search_svm.best_params_)
print("Best inner CV ROC AUC:", search_svm.best_score_)

best_svm_inner = search_svm.best_estimator_

# =========================================================
# 4. Threshold tuning on validation set (Youden / balanced acc)
# =========================================================
def tune_threshold_from_scores(scores, y_true):
    """
    Scan thresholds over the score range and pick the one
    that maximises balanced accuracy.
    """
    scores = np.asarray(scores)
    y_true = np.asarray(y_true)

    thr_grid = np.linspace(scores.min(), scores.max(), 201)
    best_thr = 0.0
    best_bal = -1.0

    for thr in thr_grid:
        y_pred = (scores >= thr).astype(int)
        bal = balanced_accuracy_score(y_true, y_pred)
        if bal > best_bal:
            best_bal = bal
            best_thr = thr

    return best_thr, best_bal

# decision_function gives continuous margins (good for threshold tuning)
scores_val = best_svm_inner.decision_function(X_val)
best_thr, best_bal_val = tune_threshold_from_scores(scores_val, y_val)

print(f"\nBest SVM threshold on validation (by balanced accuracy): {best_thr:.4f}")
print(f"Validation balanced accuracy at best threshold: {best_bal_val:.4f}")

y_val_pred_tuned = (scores_val >= best_thr).astype(int)

print("\n--- Validation metrics at tuned threshold ---")
print("ROC AUC (val):", roc_auc_score(y_val, scores_val))
print("PR AUC (val):", average_precision_score(y_val, scores_val))
print("Balanced accuracy (val):", balanced_accuracy_score(y_val, y_val_pred_tuned))
print("Confusion matrix (val):\n", confusion_matrix(y_val, y_val_pred_tuned))
print("Classification report (val):\n", classification_report(y_val, y_val_pred_tuned))

# =========================================================
# 5. Refit SVM on full training data (inner + val) and evaluate on test
# =========================================================
best_svm_final = search_svm.best_estimator_
best_svm_final.fit(X_train, y_train)   # X_train = inner + val

scores_test = best_svm_final.decision_function(X_test)

# Default 0-threshold (what .predict() uses)
y_test_pred_default = best_svm_final.predict(X_test)

# Tuned threshold from validation
y_test_pred_tuned = (scores_test >= best_thr).astype(int)

print("\n=== SVM Test Performance (default decision threshold) ===")
print("Test ROC AUC:", roc_auc_score(y_test, scores_test))
print("Test PR AUC:", average_precision_score(y_test, scores_test))
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_default))
print("Confusion matrix:\n", confusion_matrix(y_test, y_test_pred_default))
print("Classification report:\n", classification_report(y_test, y_test_pred_default))

print("\n=== SVM Test Performance (tuned threshold from validation) ===")
print("Threshold used:", best_thr)
print("Test balanced accuracy:", balanced_accuracy_score(y_test, y_test_pred_tuned))
print("Confusion matrix (tuned):\n", confusion_matrix(y_test, y_test_pred_tuned))
print("Classification report (tuned):\n", classification_report(y_test, y_test_pred_tuned))



Fitting RandomizedSearchCV for SVM on inner training data...
Fitting 25 folds for each of 50 candidates, totalling 1250 fits

Best SVM params: {'svm__C': np.float64(4.418441521199722), 'svm__class_weight': None, 'svm__gamma': np.float64(0.017885301261862014)}
Best inner CV ROC AUC: 0.6703790849673201

Best SVM threshold on validation (by balanced accuracy): -1.1783
Validation balanced accuracy at best threshold: 0.6932

--- Validation metrics at tuned threshold ---
ROC AUC (val): 0.6515151515151516
PR AUC (val): 0.3165608352685465
Balanced accuracy (val): 0.6931818181818181
Confusion matrix (val):
 [[17 27]
 [ 0 12]]
Classification report (val):
               precision    recall  f1-score   support

           0       1.00      0.39      0.56        44
           1       0.31      1.00      0.47        12

    accuracy                           0.52        56
   macro avg       0.65      0.69      0.51        56
weighted avg       0.85      0.52      0.54        56


=== SVM Test Per

In [32]:
# =========================================================
# 7. Bootstrap AUC + 95% CI helper
# =========================================================
t = '''
def bootstrap_auc_ci(y_true, scores, n_bootstrap=2000, alpha=0.95, random_state=42):
    """
    Compute AUC and (alpha*100)% CI via bootstrap.
    """
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)
    rng = np.random.default_rng(random_state)

    aucs = []
    n = len(y_true)
    for _ in range(n_bootstrap):
        idx = rng.integers(0, n, n)
        # Need both classes in the sample
        if len(np.unique(y_true[idx])) < 2:
            continue
        aucs.append(roc_auc_score(y_true[idx], scores[idx]))

    aucs = np.array(aucs)
    auc_point = roc_auc_score(y_true, scores)

    lower = np.percentile(aucs, (1 - alpha) / 2 * 100)
    upper = np.percentile(aucs, (1 + alpha) / 2 * 100)
    return auc_point, lower, upper

# Overall cohort CI as well (optional)
overall_auc, overall_low, overall_up = bootstrap_auc_ci(y_test, scores_test)
print(f"\nOverall test AUC (bootstrap): {overall_auc:.3f} "
      f"(95% CI {overall_low:.3f}–{overall_up:.3f})")

# =========================================================
# 8. HR+/HER2− subgroup AUC + 95% CI
# =========================================================
# Build DataFrame to slice subgroups on original test features
test_results = X_test.copy()
test_results["y_true"] = y_test
test_results["score"] = scores_test

# IMPORTANT: adjust the strings below to match your actual coding
# e.g. "Positive"/"Negative", "Pos"/"Neg", "1"/"0", etc.
mask_hrpos_her2neg = (
    (test_results["ER"] == "Positive") &
    (test_results["HER2"] == "Negative")
)

sub_hrpos_her2neg = test_results[mask_hrpos_her2neg]
print("\nSubgroup size (HR+/HER2-):", len(sub_hrpos_her2neg))

if sub_hrpos_her2neg["y_true"].nunique() >= 2 and len(sub_hrpos_her2neg) > 10:
    y_sub = sub_hrpos_her2neg["y_true"].values
    scores_sub = sub_hrpos_her2neg["score"].values

    auc_sub, low_sub, up_sub = bootstrap_auc_ci(y_sub, scores_sub)

    print(
        f"AUC in HR+/HER2- subgroup: {auc_sub:.3f} "
        f"(95% CI {low_sub:.3f}–{up_sub:.3f})"
    )
else:
    print("Not enough samples / class variety in HR+/HER2- subgroup to compute AUC.")

# =========================================================
# 9. Generic helper: AUC + 95% CI by levels of a single feature
# =========================================================
def subgroup_auc_table(X_test, y_test, best_model, feature_name):
    """
    For each level of a categorical feature, compute:
    - N
    - AUC
    - 95% bootstrap CI
    Returns a DataFrame.
    """
    scores = best_model.decision_function(X_test)

    df_sub = X_test.copy()
    df_sub["y_true"] = y_test
    df_sub["score"] = scores

    levels = df_sub[feature_name].unique()
    rows = []

    for lvl in levels:
        mask = (df_sub[feature_name] == lvl)
        sub = df_sub[mask]

        if sub["y_true"].nunique() < 2 or len(sub) < 10:
            # cannot compute AUC if only one class or too few samples
            continue

        y_sub = sub["y_true"].values
        scores_sub = sub["score"].values

        auc_point, low, up = bootstrap_auc_ci(y_sub, scores_sub)

        rows.append({
            feature_name: lvl,
            "n": len(sub),
            "AUC": auc_point,
            "CI_lower": low,
            "CI_upper": up,
        })

    return pd.DataFrame(rows)

# Example: performance by HER2 and ER status
her2_auc_df = subgroup_auc_table(X_test, y_test, best_svm, "HER2")
er_auc_df = subgroup_auc_table(X_test, y_test, best_svm, "ER")

print("\n=== AUC by HER2 status ===")
print(her2_auc_df)

print("\n=== AUC by ER status ===")
print(er_auc_df)
'''